In [8]:
import pandas as pd
import numpy as np

data = pd.read_csv('time_series_60min_singleindex.csv')
data.head()

,utc_timestamp,cet_cest_timestamp,AT_load_actual_entsoe_transparency,AT_load_forecast_entsoe_transparency,AT_price_day_ahead,AT_solar_generation_actual,AT_wind_onshore_generation_actual,BE_load_actual_entsoe_transparency,BE_load_forecast_entsoe_transparency,BE_solar_generation_actual,...,SI_load_actual_entsoe_transparency,SI_load_forecast_entsoe_transparency,SI_solar_generation_actual,SI_wind_onshore_generation_actual,SK_load_actual_entsoe_transparency,SK_load_forecast_entsoe_transparency,SK_solar_generation_actual,SK_wind_onshore_generation_actual,UA_load_actual_entsoe_transparency,UA_load_forecast_entsoe_transparency
0,2014-12-31T23:00:00Z,2015-01-01T00:00:00+0100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-01T00:00:00Z,2015-01-01T01:00:00+0100,5946.0,6701.0,35.0,NaN,69.0,9484.0,9897.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-01T01:00:00Z,2015-01-01T02:00:00+0100,5726.0,6593.0,45.0,NaN,64.0,9152.0,9521.0,NaN,...,1045.47,816.0,NaN,1.17,2728.0,2860.0,3.8,NaN,NaN,NaN
3,2015-01-01T02:00:00Z,2015-01-01T03:00:00+0100,5347.0,6482.0,41.0,NaN,65.0,8799.0,9135.0,NaN,...,1004.79,805.0,NaN,1.04,2626.0,2810.0,3.8,NaN,NaN,NaN
4,2015-01-01T03:00:00Z,2015-01-01T04:00:00+0100,5249.0,6454.0,38.0,NaN,64.0,8567.0,8909.0,NaN,...,983.79,803.0,NaN,1.61,2618.0,2780.0,3.8,NaN,NaN,NaN


In [4]:

date = data.loc[:, data.columns.str.startswith("utc_timestamp")]

dataDK = data.loc[:, data.columns.str.startswith("DK")]

In [5]:
newData = pd.concat([date, dataDK], axis=1)
newData.head()

,utc_timestamp,DK_load_actual_entsoe_transparency,DK_load_forecast_entsoe_transparency,DK_solar_capacity,DK_solar_generation_actual,DK_wind_capacity,DK_wind_generation_actual,DK_wind_offshore_capacity,DK_wind_offshore_generation_actual,DK_wind_onshore_capacity,...,DK_1_wind_generation_actual,DK_1_wind_offshore_generation_actual,DK_1_wind_onshore_generation_actual,DK_2_load_actual_entsoe_transparency,DK_2_load_forecast_entsoe_transparency,DK_2_price_day_ahead,DK_2_solar_generation_actual,DK_2_wind_generation_actual,DK_2_wind_offshore_generation_actual,DK_2_wind_onshore_generation_actual
0,2014-12-31T23:00:00Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-01T00:00:00Z,NaN,NaN,489.0,NaN,4643.0,NaN,1264.0,NaN,3379.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-01T01:00:00Z,3100.02,3126.8,489.0,NaN,4643.0,2357.33,1264.0,902.71,3379.0,...,1774.05,567.07,1206.98,1305.06,1341.9,16.04,NaN,583.28,335.64,247.64
3,2015-01-01T02:00:00Z,2980.39,3019.0,489.0,NaN,4643.0,2387.35,1264.0,830.87,3379.0,...,1902.23,549.83,1352.40,1235.63,1284.5,14.60,NaN,485.12,281.04,204.08
4,2015-01-01T03:00:00Z,2933.49,2976.3,489.0,NaN,4643.0,2594.47,1264.0,915.43,3379.0,...,2123.97,660.47,1463.50,1190.32,1242.3,14.95,NaN,470.50,254.96,215.54


In [18]:
cols = ['utc_timestamp', 'DK_load_actual_entsoe_transparency']

# Ensure timestamp column is datetime
newData['utc_timestamp'] = pd.to_datetime(newData['utc_timestamp'])
year_series = newData['utc_timestamp'].dt.year

years = [2015, 2016, 2017, 2018]
yearly_dataframes = {
    year: newData.loc[year_series == year, cols].copy().reset_index(drop=True)
    for year in years
}

# Optional separate variables for convenience
data_2015 = yearly_dataframes[2015]
data_2016 = yearly_dataframes[2016]
data_2017 = yearly_dataframes[2017]
data_2018 = yearly_dataframes[2018]

{year: df.shape for year, df in yearly_dataframes.items()}

{2015: (8760, 2), 2016: (8784, 2), 2017: (8760, 2), 2018: (8760, 2)}

In [32]:
# Load Renewables.ninja files (skip metadata lines that start with '#')
solar_raw = pd.read_csv('ninja_pv.csv', comment='#')
wind_raw = pd.read_csv('ninja_wind.csv', comment='#')

def prepare_cf_dataframe(df, prefix):
    cols = df.columns.astype(str).str.strip()
    df = df.copy()
    df.columns = cols

    # Find a timestamp-like column robustly
    if 'utc_timestamp' in df.columns:
        time_col = 'utc_timestamp'
    elif 'time' in df.columns:
        time_col = 'time'
    else:
        time_like = [c for c in df.columns if 'time' in c.lower() or 'date' in c.lower()]
        time_col = time_like[0] if time_like else df.columns[0]

    ts = pd.to_datetime(df[time_col], utc=True, errors='coerce').rename('utc_timestamp')
    numeric = df.drop(columns=[time_col]).apply(pd.to_numeric, errors='coerce')
    numeric = numeric.add_prefix(prefix)

    out = pd.concat([ts, numeric], axis=1)
    out = out.dropna(subset=['utc_timestamp']).sort_values('utc_timestamp').reset_index(drop=True)
    return out

solar_cf = prepare_cf_dataframe(solar_raw, 'pv_cf_')
wind_cf = prepare_cf_dataframe(wind_raw, 'wind_cf_')

solar_cf_cols = [c for c in solar_cf.columns if c != 'utc_timestamp']
wind_cf_cols = [c for c in wind_cf.columns if c != 'utc_timestamp']

solar_yearly_dataframes = {
    year: solar_cf.loc[solar_cf['utc_timestamp'].dt.year == year, ['utc_timestamp'] + solar_cf_cols]
    .reset_index(drop=True)
    .copy()
    for year in years
}

wind_yearly_dataframes = {
    year: wind_cf.loc[wind_cf['utc_timestamp'].dt.year == year, ['utc_timestamp'] + wind_cf_cols]
    .reset_index(drop=True)
    .copy()
    for year in years
}

# Optional separate variables
solar_2015 = solar_yearly_dataframes[2015]
solar_2016 = solar_yearly_dataframes[2016]
solar_2017 = solar_yearly_dataframes[2017]
solar_2018 = solar_yearly_dataframes[2018]

wind_2015 = wind_yearly_dataframes[2015]
wind_2016 = wind_yearly_dataframes[2016]
wind_2017 = wind_yearly_dataframes[2017]
wind_2018 = wind_yearly_dataframes[2018]

print('PV capacity factor columns:', solar_cf_cols)
print('Wind capacity factor columns:', wind_cf_cols)
print({year: (solar_yearly_dataframes[year].shape, wind_yearly_dataframes[year].shape) for year in years})

/var/folders/np/y0lp2y655xv_szx354zcxsmh0000gn/T/ipykernel_16573/916375726.py:2: DtypeWarning: Columns (0: Unnamed: 1, 1: Unnamed: 2, 2: Unnamed: 3, 3: Unnamed: 4, 4: Unnamed: 5) have mixed types. Specify dtype option on import or set low_memory=False.
  solar_raw = pd.read_csv('ninja_pv.csv', comment='#')
/var/folders/np/y0lp2y655xv_szx354zcxsmh0000gn/T/ipykernel_16573/916375726.py:3: DtypeWarning: Columns (0: Unnamed: 1, 1: Unnamed: 2, 2: Unnamed: 3, 3: Unnamed: 4, 4: Unnamed: 5, 5: Unnamed: 6) have mixed types. Specify dtype option on import or set low_memory=False.
  wind_raw = pd.read_csv('ninja_wind.csv', comment='#')
/var/folders/np/y0lp2y655xv_szx354zcxsmh0000gn/T/ipykernel_16573/916375726.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ts = pd.to_datetime(df[time_col], utc=True, errors='coerce').rename('utc_timestamp')
/var/folders/np

PV capacity factor columns: ['pv_cf_Unnamed: 1', 'pv_cf_Unnamed: 2', 'pv_cf_Unnamed: 3', 'pv_cf_Unnamed: 4', 'pv_cf_Unnamed: 5']
Wind capacity factor columns: ['wind_cf_Unnamed: 1', 'wind_cf_Unnamed: 2', 'wind_cf_Unnamed: 3', 'wind_cf_Unnamed: 4', 'wind_cf_Unnamed: 5', 'wind_cf_Unnamed: 6']
{2015: ((8760, 6), (8760, 7)), 2016: ((8784, 6), (8784, 7)), 2017: ((8760, 6), (8760, 7)), 2018: ((8760, 6), (8760, 7))}


In [36]:
import matplotlib.pyplot as plt

# Keep first 2 columns (utc_timestamp + one data column) for solar and wind datasets
solar_2015_slim = solar_2015.iloc[:, :2].copy()
solar_2016_slim = solar_2016.iloc[:, :2].copy()
solar_2017_slim = solar_2017.iloc[:, :2].copy()
solar_2018_slim = solar_2018.iloc[:, :2].copy()

wind_2015_slim = wind_2015.iloc[:, :2].copy()
wind_2016_slim = wind_2016.iloc[:, :2].copy()
wind_2017_slim = wind_2017.iloc[:, :2].copy()
wind_2018_slim = wind_2018.iloc[:, :2].copy()

# Merge and concatenate for each year
merged_2015 = data_2015.merge(solar_2015_slim, on='utc_timestamp', how='inner').merge(wind_2015_slim, on='utc_timestamp', how='inner')
merged_2016 = data_2016.merge(solar_2016_slim, on='utc_timestamp', how='inner').merge(wind_2016_slim, on='utc_timestamp', how='inner')
merged_2017 = data_2017.merge(solar_2017_slim, on='utc_timestamp', how='inner').merge(wind_2017_slim, on='utc_timestamp', how='inner')
merged_2018 = data_2018.merge(solar_2018_slim, on='utc_timestamp', how='inner').merge(wind_2018_slim, on='utc_timestamp', how='inner')

# Save to CSV files
merged_2015.to_csv('DK_2015_merged.csv', index=False)
merged_2016.to_csv('DK_2016_merged.csv', index=False)
merged_2017.to_csv('DK_2017_merged.csv', index=False)
merged_2018.to_csv('DK_2018_merged.csv', index=False)

print("Files saved successfully:")
print(f"DK_2015_merged.csv - {merged_2015.shape}")
print(f"DK_2016_merged.csv - {merged_2016.shape}")
print(f"DK_2017_merged.csv - {merged_2017.shape}")
print(f"DK_2018_merged.csv - {merged_2018.shape}")

Files saved successfully:
DK_2015_merged.csv - (8760, 4)
DK_2016_merged.csv - (8784, 4)
DK_2017_merged.csv - (8760, 4)
DK_2018_merged.csv - (8760, 4)
